
# Mouse Alignment Analysis — STAR All 26 (`GRCm39` + Ensembl)

This notebook extends the Trapnell-style alignment walkthrough into a more organized, graduate-level mouse report. It focuses on the canonical `fastp`-cleaned alignment run for all 26 samples and asks three practical questions:

1. Are the STAR alignment metrics strong enough to proceed with the full dataset?
2. Do the remaining GC-WARN samples behave differently from the GC-PASS samples at the alignment stage?
3. Is there visible structure by sequencing platform (`NovaSeq 6000` vs `NovaSeq X`) or biological subgroup that should be carried forward into downstream interpretation?

The notebook parses the local copy of the canonical STAR outputs, caches sample metadata from GEO, builds count-matrix handoff artifacts from `ReadsPerGene.out.tab`, and saves cleaned tables/figures for the individual report and later DE work.


In [ ]:

from pathlib import Path
import re
import time
import urllib.request
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 200)
plt.rcParams['figure.dpi'] = 120



## Data locations and provenance

The canonical local alignment copy lives under the mouse project tree and mirrors the private server-side STAR run. This notebook writes all alignment-analysis artifacts into a dedicated local output folder so the notebook remains reproducible and the deliverables are easy to reuse.


In [ ]:

MOUSE_ROOT = Path('/Users/pitergarcia/DataScience/Semester5/BIOL550/group_project/mouse')
ALIGN_ROOT = MOUSE_ROOT / 'alignment_local_server_private_copy' / 'star_grcm39_ensembl_all26_fastp'
SAMPLES_DIR = ALIGN_ROOT / 'samples'
RUNINFO_CSV = Path('/Users/pitergarcia/DataScience/Semester5/BIOL550/BIOL550-Lab/project_pic/project_datasets/PRJNA1017789_runinfo.csv')
GC_HEATMAP = MOUSE_ROOT / 'qc_analysis_remediation' / 'multiqc_fastp_trim_only_shared' / 'mouse_fastp_trim_only_multiqc_data' / 'fastqc-status-check-heatmap.txt'
OUT_DIR = MOUSE_ROOT / 'alignment_analysis_star_all26'
FIG_DIR = OUT_DIR / 'figures'
TABLE_DIR = OUT_DIR / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

for path in [ALIGN_ROOT, SAMPLES_DIR, RUNINFO_CSV, GC_HEATMAP]:
    print(f'{path}:', 'OK' if path.exists() else 'MISSING')


In [ ]:

sample_dirs = sorted([p for p in SAMPLES_DIR.iterdir() if p.is_dir()])
log_files = sorted(SAMPLES_DIR.glob('*/SRR*.Log.final.out'))
reads_files = sorted(SAMPLES_DIR.glob('*/SRR*.ReadsPerGene.out.tab'))
bam_files = sorted(SAMPLES_DIR.glob('*/*.bam'))
bai_files = sorted(BAM_FILES if 'BAM_FILES' in globals() else [])

copy_status = pd.DataFrame([
    {'artifact': 'sample directories', 'expected': 26, 'found': len(sample_dirs)},
    {'artifact': 'Log.final.out files', 'expected': 26, 'found': len(log_files)},
    {'artifact': 'ReadsPerGene.out.tab files', 'expected': 26, 'found': len(reads_files)},
    {'artifact': 'BAM files', 'expected': 26, 'found': len(bam_files)},
    {'artifact': 'BAM index files', 'expected': 26, 'found': len(bai_files)},
])
copy_status['status'] = np.where(copy_status['found'] >= copy_status['expected'], 'complete', 'incomplete')
display(copy_status)

if len(bam_files) < 26:
    display(Markdown(
        f"**Note:** BAM sync is still incomplete locally (`{len(bam_files)}/26` BAMs present). "
        "The STAR log/count analysis below is still valid because it uses `Log.final.out` and `ReadsPerGene.out.tab`, "
        "but any BAM-level follow-up should wait until the file transfer finishes."
    ))



## Reference and alignment provenance

This run uses the canonical `fastp`-cleaned inputs and the locally selected mouse reference pair (`GRCm39` primary assembly + matching Ensembl annotation). The table below comes from the alignment run metadata recorded when the all-26 STAR run was launched.
